In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt
/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json
/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt
/kaggle/input/datasets/mahasaad12/simulation/compile.log
/kaggle/input/datasets/mahasaad12/simulation/counter_asseration.sv
/kaggle/input/datasets/mahasaad12/simulation/counter.sv
/kaggle/input/datasets/mahasaad12/simulation/coverage.sv
/kaggle/input/datasets/mahasaad12/simulation/Test_counter.sv


In [11]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [13]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# LOAD DOUCMENTS 

In [15]:
from langchain_community.document_loaders import TextLoader
spec = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt").load()

interface = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt").load()

rtl = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json").load()

tb = TextLoader("/kaggle/input/datasets/mahasaad12/simulation/Test_counter.sv").load()

assertions = TextLoader("/kaggle/input/datasets/mahasaad12/simulation/counter_asseration.sv").load()

coverage = TextLoader("/kaggle/input/datasets/mahasaad12/simulation/coverage.sv").load()

compile_log =TextLoader("/kaggle/input/datasets/mahasaad12/simulation/compile.log").load()

documents = spec + interface + rtl+ tb + assertions + coverage + compile_log 

print(documents)


[Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt'}, page_content='Counter Specification\n\nThe counter increments on every rising edge of the clock.\n\nMaximum value = 15.\n\nAfter reaching 15, the counter wraps back to zero.\n\nReset initializes the counter to zero.'), Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt'}, page_content='Inputs\n\nclk\n\nrst\n\nOutputs\n\ncount[3:0]'), Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json'}, page_content='{\n    "module":"counter",\n\n    "inputs":[\n        "clk",\n        "rst"\n    ],\n\n    "outputs":[\n        "count"\n    ],\n\n    "registers":[\n        "count"\n    ],\n\n    "always_blocks":1\n}'), Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/simulation/Test_counter.sv'}, page_content="module counter_tb();\n\n\n    // Declare all signals\n    logic clk;\n    logic 

# SPLITT LOAD EMBEDDING MODEL

In [21]:


text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(chunks, embedding)

In [24]:
import json

with open("/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt") as f:
    spec = f.read()

with open("/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json") as f:
    rtl = json.load(f)

with open("/kaggle/input/datasets/mahasaad12/simulation/counter_asseration.sv") as f:
    assertions = f.read()

with open("/kaggle/input/datasets/mahasaad12/simulation/Test_counter.sv") as f:
    tb = f.read()

with open("/kaggle/input/datasets/mahasaad12/simulation/coverage.sv") as f:
    coverage = f.read()

with open("/kaggle/input/datasets/mahasaad12/simulation/compile.log") as f:
    compile_log = f.read()

with open("/kaggle/input/datasets/mahasaad12/simulation/coverage.sv") as f:
    coverage_report = f.read()

In [28]:
context = f"""

Specification

{spec}

RTL

{json.dumps(rtl, indent=4)}

Assertions

{assertions}

Coverage

{coverage}

Compile Log

{compile_log}

Coverage Report

{coverage_report}

"""

In [30]:
prompt = f"""
You are a Senior ASIC Verification Engineer.

Your task is to analyze the verification results.

Context

{context}

Tasks

1. Identify every failure.

2. Explain why it happened.

3. Determine whether the issue is:

- RTL bug
- Assertion bug
- Testbench bug
- Coverage gap
- Compilation error

4. Explain the evidence.

5. Suggest a fix.

6. Suggest an additional verification test.

Return the answer as Markdown.
"""

In [31]:
answer = generate_text(prompt)

print(answer)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a Senior ASIC Verification Engineer.

Your task is to analyze the verification results.

Context



Specification

Counter Specification

The counter increments on every rising edge of the clock.

Maximum value = 15.

After reaching 15, the counter wraps back to zero.

Reset initializes the counter to zero.

RTL

{
    "module": "counter",
    "inputs": [
        "clk",
        "rst"
    ],
    "outputs": [
        "count"
    ],
    "registers": [
        "count"
    ],
    "always_blocks": 1
}

Assertions

module counter_assertions;

  input clk, rst;
  output reg [3:0] count;

  property counter_property;
    @(posedge clk)
    $stable(count) || (count == 0);
  endproperty

  assert property_counter {
    counter_property;
  }

  property counter_reset;
    rst = 1'b1 => count == 0;
  endproperty

  assert property_counter_wraparound {
    $past(count) == 15 => count == 0;
  }

  property counter_max_value {
    count <= 15;
  }

  property counter_always_zero_on_reset {
  